# DG-TCAV Kaggle launcher
Attach authorized private preprocessed Cohort A data and trusted MedicalNet weights. Enable a GPU for real training. This notebook launches repository code; it does not perform preprocessing.

In [ ]:
!git clone --branch classifier-implementation https://github.com/Alishals28/dg-tcav-classifier-fyp.git
%cd dg-tcav-classifier-fyp
!pip install -r requirements.txt
!pip install -e . --no-deps

In [ ]:
import torch
print('PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
!git rev-parse HEAD
!python -m pytest

## Configure your actual inputs
Edit configs/kaggle.yaml following the README. Set manifest_path, splits_path, volume_dir, file_pattern, reference_image, orientation, normalization, and the pretrained checkpoint path. Set preprocessing_confirmed=true **only after receiving QC approval**. The default deliberately refuses real training.

In [ ]:
# The next commands require completed configuration and preprocessing.
!python -m scripts.validate_dataset --config configs/kaggle.yaml
!python -m scripts.tiny_overfit --config configs/kaggle.yaml --output /kaggle/working/tiny_overfit

Run a separate 1–2 epoch real-data smoke config first: full width, smoke_test=false, epochs=2, new experiment name. Measure GPU memory and epoch time before the main run.

In [ ]:
!python -m src.train --config configs/kaggle.yaml

## After freezing all model choices
Copy the exact printed run path. Final test is intentionally commented out. Preserve/download the entire run directory before session expiry.

In [ ]:
# !python -m src.evaluate --config configs/kaggle.yaml --checkpoint /kaggle/working/outputs/EXACT_RUN/best.pt --split test --confirm-final-test
# !python -m src.export_activations --config configs/kaggle.yaml --checkpoint /kaggle/working/outputs/EXACT_RUN/best.pt --output /kaggle/working/features